In [ ]:
!pip -q install mediapipe opencv-python


In [15]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install -U mediapipe opencv-python

import cv2
import mediapipe as mp
import os, json
import datetime
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# ======================
# Paths (Drive)
# ======================
project_folder_name = 'datapingpong-vision-lab/03_pose-estimation'
project_path = os.path.join('/content/drive/MyDrive', project_folder_name)

video_path = os.path.join(project_path, "../01_ball-tracking/data/DJI_0056_001.MP4")

out_dir = os.path.join(project_path, "output")
os.makedirs(out_dir, exist_ok=True)
out_jsonl_path = os.path.join(out_dir, "pose_keypoints.jsonl")

# ======================
# Download pose model (.task)
# ======================
model_path = "/content/pose_landmarker_lite.task"
if not os.path.exists(model_path):
    !wget -q -O /content/pose_landmarker_lite.task \
      https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task

# ======================
# Create PoseLandmarker (VIDEO) : left/right separate to avoid timestamp collision
# ======================
options = vision.PoseLandmarkerOptions(
    base_options=python.BaseOptions(model_asset_path=model_path),
    running_mode=vision.RunningMode.VIDEO,
    num_poses=1,
)
landmarker_left  = vision.PoseLandmarker.create_from_options(options)
landmarker_right = vision.PoseLandmarker.create_from_options(options)

# ======================
# Video open
# ======================
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    raise RuntimeError(f"Failed to open video: {video_path}")

fps = cap.get(cv2.CAP_PROP_FPS)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
mid = w // 2

def landmarks_to_abs_list(landmarks_norm, roi_x0, roi_w, roi_h):
    """
    landmarks_norm: list of NormalizedLandmark (x,y,z,visibility)
    Convert to absolute pixel coords in full frame with x offset.
    """
    out = []
    for i, lm in enumerate(landmarks_norm):
        x_abs = float(lm.x * roi_w + roi_x0)
        y_abs = float(lm.y * roi_h)
        z_val = float(lm.z) if hasattr(lm, "z") else 0.0
        vis = float(lm.visibility) if hasattr(lm, "visibility") else None
        out.append({"id": i, "x": x_abs, "y": y_abs, "z": z_val, "vis": vis})
    return out

# ======================
# Process & write JSONL
# ======================
written = 0
frame_idx = 0

with open(out_jsonl_path, "w", encoding="utf-8") as f:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        left_bgr = frame[:, :mid]
        right_bgr = frame[:, mid:]

        left_rgb = cv2.cvtColor(left_bgr, cv2.COLOR_BGR2RGB)
        right_rgb = cv2.cvtColor(right_bgr, cv2.COLOR_BGR2RGB)

        left_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=left_rgb)
        right_img = mp.Image(image_format=mp.ImageFormat.SRGB, data=right_rgb)

        ts_ms = int(frame_idx * 1000 / fps)

        res_l = landmarker_left.detect_for_video(left_img, ts_ms)
        res_r = landmarker_right.detect_for_video(right_img, ts_ms)

        rec = {
            "frame": frame_idx,
            "t_ms": ts_ms,
            "fps": float(fps),
            "size": {"w": w, "h": h},
            "left":  {"roi": {"x0": 0,   "y0": 0, "w": mid,     "h": h}, "landmarks": None},
            "right": {"roi": {"x0": mid, "y0": 0, "w": w - mid, "h": h}, "landmarks": None},
        }

        if res_l.pose_landmarks and len(res_l.pose_landmarks) > 0:
            rec["left"]["landmarks"] = landmarks_to_abs_list(res_l.pose_landmarks[0], roi_x0=0, roi_w=mid, roi_h=h)

        if res_r.pose_landmarks and len(res_r.pose_landmarks) > 0:
            rec["right"]["landmarks"] = landmarks_to_abs_list(res_r.pose_landmarks[0], roi_x0=mid, roi_w=w-mid, roi_h=h)

        f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        written += 1
        frame_idx += 1

        if frame_idx % 300 == 0:
          print(datetime.datetime.now(), "frame_idx:", frame_idx)

cap.release()

print("saved jsonl:", out_jsonl_path)
print("frames written:", written)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
2026-02-01 05:54:29.001169 frame_idx: 300
2026-02-01 05:54:52.352658 frame_idx: 600
2026-02-01 05:55:18.764827 frame_idx: 900
2026-02-01 05:55:44.725067 frame_idx: 1200
2026-02-01 05:56:10.501875 frame_idx: 1500
2026-02-01 05:56:34.976779 frame_idx: 1800
2026-02-01 05:57:01.292435 frame_idx: 2100
2026-02-01 05:57:27.226926 frame_idx: 2400
2026-02-01 05:57:53.205944 frame_idx: 2700
2026-02-01 05:58:17.900239 frame_idx: 3000
2026-02-01 05:58:43.544119 frame_idx: 3300
2026-02-01 05:59:09.931863 frame_idx: 3600
2026-02-01 05:59:35.972665 frame_idx: 3900
2026-02-01 06:00:00.303083 frame_idx: 4200
2026-02-01 06:00:26.148275 frame_idx: 4500
2026-02-01 06:00:52.064632 frame_idx: 4800
2026-02-01 06:01:17.961802 frame_idx: 5100
2026-02-01 06:01:42.210325 frame_idx: 5400
2026-02-01 06:02:08.094786 frame_idx: 5700
2026-02-01 06:02:34.167300 frame_idx: 6000
2026-02-01 06:

In [ ]:
# pose_keypoints.jsonl フォーマット（1行 = 1フレーム）
#
# {
#   "frame": 123,                 # フレーム番号（0始まり）
#   "t_ms": 1025,                 # 時刻 [ms] = frame / fps * 1000
#   "fps": 120.0,                 # 動画FPS
#   "size": { "w": 1920, "h": 1080 },   # 元動画サイズ（pixel）
#
#   "left": {                     # 左プレイヤー（画面左ROI）
#     "roi": { "x0": 0, "y0": 0, "w": 960, "h": 1080 },
#     "landmarks": [              # 検出失敗時は null
#       {
#         "id": 15,               # MediaPipe Pose landmark id (0-32)
#         "x": 1342.3,            # x座標（フルフレーム基準, pixel）
#         "y": 612.8,             # y座標（フルフレーム基準, pixel）
#         "z": -0.124,            # 相対Z（単位なし, 参考値）
#         "vis": 0.91             # visibility（0-1目安）
#       }
#     ]
#   },
#
#   "right": {                    # 右プレイヤー（画面右ROI）
#     "roi": { "x0": 960, "y0": 0, "w": 960, "h": 1080 },
#     "landmarks": [ ... ] | null
#   }
# }
#
# 座標系:
# - (0,0) は左上、x右向き+, y下向き+
# - right の x もオフセット済み（左右同一座標系）
#
# 想定用途:
# - frame / t_ms で hit・bounce と同期
# - x,y で速度・角度計算
# - landmarks=null は前後フレーム補間前提
